In [6]:
# HypotheSAEs Quickstart
# This notebook demonstrates basic usage of HypotheSAEs on a sample of the Yelp review dataset
import os

%load_ext autoreload
%autoreload 2

import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # Set to '0' to use the first GPU, or 'cpu' to run on CPU
os.environ['OPENAI_KEY_SAE'] = 'EMPTY' # Replace with your OpenAI API key, or with another environment variable (e.g. os.environ['OPENAI_API_
import numpy as np
import pandas as pd

from hypothesaes.quickstart import train_sae, interpret_sae, generate_hypotheses, evaluate_hypotheses
from hypothesaes.embedding import get_openai_embeddings, get_local_embeddings

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


**Load data**

The dataset we're using here is a subset of 20K Yelp reviews, with 2K reviews used for validation (during SAE training). 

The target variable is the `stars` column, which is a rating between 1 and 5. We treat this as a regression task.

There are also 2K reviews used for holdout evaluation, which we'll use at the end of the notebook.

In [7]:
CACHE_SIGNAL = "stf"

In [8]:
current_dir = os.getcwd()
if current_dir.endswith("notebooks"):
    prefix = "../"
else:
    prefix = "./"
val_ratio = 0.1  # Ratio of training data to use for validation
import sklearn
import sklearn.model_selection
few_shot_examples = 5
# base_dir = os.path.join(prefix, "demo-data")
# train_df = pd.read_json(os.path.join(base_dir, "yelp-demo-train-20K.json"), lines=True)
# val_df = pd.read_json(os.path.join(base_dir, "yelp-demo-val-2K.json"), lines=True)

# texts = train_df['text'].tolist()
# labels = train_df['stars'].values
# val_texts = val_df['text'].tolist() # These are only used for early stopping of SAE training, so we don't need labels.
from utils import df_to_prompts
import json
import re
base_dir = os.path.join(prefix, 'stf_data')
# train_X = pd.read_csv(os.path.join(base_dir, "X_train.csv")) 
train_y = pd.read_csv(os.path.join(base_dir, "y_train.csv")).values.ravel()
# test_X = pd.read_csv(os.path.join(base_dir, "X_test.csv"))
test_y = pd.read_csv(os.path.join(base_dir, "y_test.csv")).values.ravel()
label_type = set(train_y).union(set(test_y))
# number_dict = {label: i for i, label in enumerate(label_type)}
number_dict = {label: 1 if label == 'Yes' else 0 for label in label_type}
label_train = [number_dict[label] for label in train_y]
label_test = [number_dict[label] for label in test_y]
# train_texts = df_to_prompts(few_shot_row, few_shot_label, train_X.iloc[few_shot_examples:, :], few_shot_examples=few_shot_examples)
pattern = re.compile(r"<think>.*?</think>", re.DOTALL)  # 匹配 <think> 到 </think>，包括换行
train_texts_idx_profile = []
test_texts_idx_profile = []
train_jsonl = os.path.join(base_dir, f"train_profile_{CACHE_SIGNAL}.jsonl")
test_josnl = os.path.join(base_dir, f"test_profile_{CACHE_SIGNAL}.jsonl")
if len(CACHE_SIGNAL) == 0 or 'toy' in CACHE_SIGNAL :
    train_jsonl = os.path.join(base_dir, "train_profile_toy.jsonl")
    test_josnl = os.path.join(base_dir, "test_profile_toy.jsonl")
with open(train_jsonl) as f:
    for line in f:
        obj = json.loads(line)
        profile = obj["profile"]
        profile = re.sub(pattern, "", profile)  # 删除 <think>...</think>
        obj["profile"] = profile.strip()
        train_texts_idx_profile.append((obj['profile'], obj['idx']))
    sorted_train_texts_idx_profile = sorted(train_texts_idx_profile, key=lambda x: x[1])
train_texts = [item[0] for item in sorted_train_texts_idx_profile]
with open(test_josnl)as f:
    for line in f:
        obj = json.loads(line)
        profile = obj["profile"]
        profile = re.sub(pattern, "", profile)  # 删除 <think>...</think>
        obj["profile"] = profile.strip()
        test_texts_idx_profile.append((obj['profile'], obj['idx']))
    sorted_test_texts_idx_profile = sorted(test_texts_idx_profile, key=lambda x: x[1])
test_texts = [item[0] for item in sorted_test_texts_idx_profile]
print(set(train_y), set(test_y))    
number_dict

{'No', 'Yes'} {'No', 'Yes'}


{'No': 0, 'Yes': 1}

In [9]:
#count label distribution
train_label_distribution = pd.Series(train_y).value_counts()
test_label_distribution = pd.Series(test_y).value_counts()
train_label_distribution,  test_label_distribution

(Yes    4912
 No     4912
 Name: count, dtype: int64,
 Yes    234
 No     234
 Name: count, dtype: int64)

In [10]:
texts, val_texts, labels, val_labels = sklearn.model_selection.train_test_split(
    train_texts, label_train, test_size=val_ratio, random_state=42, shuffle=True
)


In [11]:
len(test_y)

468

**Compute text embeddings for your dataset**

We'll compute text embeddings for a training set, and optionally a validation set. The validation embeddings are used for SAE eval and early-stopping during training.

Embeddings will be stored in the `emb_cache` directory (or `os.environ["EMB_CACHE_DIR"]` if you set it) using the `cache_name` parameter, so you only need to compute embeddings once.

You can use OpenAI or a local model.

Local models will run much faster on GPU. The default local model is `nomic-ai/modernbert-embed-base`. You can use any sentence-transformers model, but please read the model's docs; you may need to edit `get_local_embeddings`.

In [12]:
EMBEDDER = "Qwen/Qwen3-Embedding-0.6B" # OpenAI
# EMBEDDER = "nomic-ai/modernbert-embed-base" # Huggingface model, will run locally
CACHE_NAME = f"yelp_quickstart_{EMBEDDER}"

# text2embedding = get_openai_embeddings(texts + val_texts, model=EMBEDDER, cache_name=CACHE_NAME)
text2embedding = get_local_embeddings(texts + val_texts, model=EMBEDDER, batch_size=32, cache_name=CACHE_NAME)
embeddings = np.stack([text2embedding[text] for text in texts])

train_embeddings = np.stack([text2embedding[text] for text in texts])
val_embeddings = np.stack([text2embedding[text] for text in val_texts])
text2embedding_test = get_local_embeddings(test_texts, model=EMBEDDER, batch_size=8, cache_name=CACHE_NAME)
test_embeddings = np.stack([text2embedding_test[text] for text in test_texts])

Loading embedding chunks: 100%|██████████| 8/8 [00:00<00:00, 47.13it/s]


Loaded 13614 embeddings in 0.2s
Loaded model Qwen/Qwen3-Embedding-0.6B to cuda


Processing chunks: 100%|██████████| 1/1 [02:39<00:00, 159.30s/it]


Saved 9824 embeddings to /home/sevan/myHypotheSAEs/emb_cache/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/chunk_008.npy


Loading embedding chunks: 100%|██████████| 9/9 [00:00<00:00, 49.55it/s]


Loaded 23438 embeddings in 0.2s
Loaded model Qwen/Qwen3-Embedding-0.6B to cuda


Processing chunks: 100%|██████████| 1/1 [00:07<00:00,  7.71s/it]

Saved 468 embeddings to /home/sevan/myHypotheSAEs/emb_cache/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/chunk_009.npy


**Train SAE(s)** 

Using different values of $M$ and $k$ will produce features at different levels of granularity. You can train multiple SAEs if you'd like to produce features at varying granularity, but this is optional.

See the README for more details about selecting $M$ and $k$.

In [14]:
checkpoint_dir = os.path.join(prefix, f"checkpoints_{CACHE_SIGNAL}", CACHE_NAME)
# sae_256_8 = train_sae(embeddings=train_embeddings, M=256, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
# sae_32_4 = train_sae(embeddings=train_embeddings, M=32, K=4, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
# sae_512_8 = train_sae(embeddings=train_embeddings, M=512, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_64 = train_sae(embeddings=train_embeddings, M=128, K=8, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_16 = train_sae(embeddings=train_embeddings, M=128, K=16, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_256_16 = train_sae(embeddings=train_embeddings, M=256, K=16, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_32 = train_sae(embeddings=train_embeddings, M=128, K=32, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_256_32 = train_sae(embeddings=train_embeddings, M=256, K=32, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_512_32 = train_sae(embeddings=train_embeddings, M=512, K=32, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_128_64 = train_sae(embeddings=train_embeddings, M=128, K=64, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_256_64 = train_sae(embeddings=train_embeddings, M=256, K=64, checkpoint_dir=checkpoint_dir, val_embeddings=val_embeddings, n_epochs=1000, patience=100)
sae_list = [sae_256_32, sae_128_64, sae_256_64]

Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=8.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=16.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=256_K=16.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=32.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=256_K=32.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=512_K=32.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=128_K=64.pt onto device cuda
Loaded model from ./checkpoints_stf/yelp_quickstart_Qwen/Qwen3-Embedding-0.6B/SAE_M=256_K=64.pt onto device cuda


**Interpret neurons**  

Interpret a random subset of neurons in the SAE to sanity-check that the learned features, and their interpretations, seem reasonable. We generate and print labels for `n_random_neurons` neurons, and we also print out the top-activating texts for each neuron.

In [15]:
# This instruction will be included in the neuron interpretation prompt.
# The below instructions are specific to Yelp, but you can customize this for your task.
# If you don't pass in task-specific instructions, there is a generic instruction (see src/interpret_neurons.py);
# task-specific instructions are optional, but they help produce hypotheses at the desired level of specificity.

TASK_SPECIFIC_INSTRUCTIONS = """You are a medical assistant. You need add hypothesis for each neuron, which should describe a specific aspect of the text.
Features should describe a specific aspect of patient profile. For example:
- "mentions patient has a tumor in the left lung"
- "indicates patient is allergic to penicillin"
""" 
# """All of the texts are reviews of restaurants on Yelp.
# Features should describe a specific aspect of the review. For example:
# - "mentions long wait times to receive service"
# - "praises how a dish was cooked, with phrases like 'perfect medium-rare'\""""
# Interpret random neurons
results = interpret_sae(
    texts=texts,
    embeddings=train_embeddings,
    sae=sae_list,
    n_random_neurons=30,
    print_examples_n=10,
    task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
    interpreter_model="Qwen/Qwen3-32B",
    annotator_model="Qwen/Qwen3-32B",  # Use the same model for both interpretation and annotation
    max_interpretation_tokens= 1024,
)

Computing activations (batchsize=16384): 100%|██████████| 1/1 [00:00<00:00, 162.57it/s]


Activations shape: (8841, 640)


Generating 1 interpretation(s) per neuron: 100%|██████████| 30/30 [00:08<00:00,  3.46it/s]



Neuron 153 (from SAE M=256, K=32): <think>

</think>

- "describes a mass or lesion involving the nervous system or nerve-related structures, such as the spine, psoas muscle, sciatic nerve, or neural foramen

Top activating examples:
1. A 55-year-old woman with type 2 diabetes mellitus, hypertension, and hyperlipidemia presented with an 11-day history of a pruritic, painful, blistering rash on her trunk, face, and extremities, without mucosal involvement. She had started naltrexone for alcohol use disorder 3 days before rash onset and denied any other new medications or supplements. Initially, she was admitted elsewhere for fever and a perioral rash and treated with antibiotics and steroids for one week. Two days after discharge, she developed a vesiculobullous eruption that progressed to her trunk and extremities, prompting transfer for further evaluation.  On examination, she was afebrile and tachycardic. The skin showed diffuse erythema with bullae and pustules forming dependent pu

**Generate hypotheses**

Generate hypotheses which are predictive of the target variable.

The `selection_method` parameter defines how we compute neuron predictiveness (see `src/select_neurons.py` for more details):
- "separation_score": E[target | top-activating examples] - E[target | zero-activating examples]
- "correlation": pearson(neuron activations, target variable)
- "lasso": select N nonzero features with an L1 regularized model

This cell outputs a dataframe with the following columns:
- `neuron_idx`: The index of the neuron in the SAE (if you're using multiple SAEs, this will be a global index across all of them).
- `source_sae`: The SAE that the neuron was selected from.
- `target_{selection_method}`: The predictiveness of the neuron for the target variable, using the selected `selection_method`.
- `interpretation`: The natural language interpretation of the neuron.
- `interp_fidelity_score`: The F1 fidelity score for how well the neuron's interpretation actually corresponds to its activation pattern.

In [28]:
RESULT_EXTRA_INFO = "20_Hypotheses"

In [29]:
selection_method = "lasso"
multi_class = True  # Set to True for direct multi-class classification, False for using one vs. rest classification
if len(set(labels)) == 2:
    selection_method = "lasso"  # Use logistic regression for binary classification
    multi_class = True
result_dir = f'result_cache/{CACHE_SIGNAL}_one_vs_rest_{RESULT_EXTRA_INFO}' if not multi_class else f'result_cache/{CACHE_SIGNAL}_multi_class_{RESULT_EXTRA_INFO}'

if not os.path.exists(result_dir):
    os.makedirs(result_dir)
if multi_class:
    results = generate_hypotheses(
        texts=texts,
        labels=labels,
        embeddings=embeddings,
        sae=sae_list,
        cache_name=CACHE_NAME,
        selection_method=selection_method,
        n_selected_neurons=20,
        n_candidate_interpretations=20,
        task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
        classification=True,
        interpreter_model="Qwen/Qwen3-32B",
        annotator_model="Qwen/Qwen3-32B",  # Use the same model for both interpretation and annotation
    )

    print("\nMost predictive features of Yelp reviews:")
    pd.set_option('display.max_colwidth', None)
    display(results.sort_values(by=f"target_{selection_method}", ascending=False))
    pd.reset_option('display.max_colwidth')
    results.to_csv(os.path.join(result_dir, f"hypotheses_{selection_method}.csv"), index=False)
else:
    results = {}
    for key in number_dict.keys():
        converted_key = {number_dict[dic_key]: 1 if dic_key == key else 0 for dic_key in number_dict.keys()}
        converted_label = [converted_key[label] for label in labels]
        print(len(converted_label))
        results[f'{key}_vs_rest'] = generate_hypotheses(
            texts=texts,
            labels=converted_label,
            embeddings=embeddings,
            sae=sae_list,
            cache_name=CACHE_NAME,
            selection_method=selection_method,
            n_selected_neurons=15,
            n_candidate_interpretations=3,
            task_specific_instructions=TASK_SPECIFIC_INSTRUCTIONS,
            classification=True,
            n_scoring_examples= 1000,  # Number of examples to use for scoring hypotheses
            interpreter_model="Qwen/Qwen3-32B",
            annotator_model="Qwen/Qwen3-32B",  # Use the same model for both interpretation and annotation
        )
    for key, value in results.items():
        key_norm = key.replace("/", "or")
        key_norm = key_norm.replace(" ", "_")
        print(f"\nMost predictive features of Yelp reviews for {key}:")
        pd.set_option('display.max_colwidth', None)
        display(value.sort_values(by=f"target_{selection_method}", ascending=False))
        pd.reset_option('display.max_colwidth')
        results[key].to_csv(os.path.join(result_dir, f"hypotheses_{key_norm}_{selection_method}.csv"), index=False)
        

Embeddings shape: (8841, 1024)


Computing activations (batchsize=16384): 100%|██████████| 1/1 [00:00<00:00, 178.90it/s]

Activations shape: (8841, 640)

Step 1: Selecting top 20 predictive neurons
LASSO iteration   L1 Alpha # Features   Time (s)
----------------------------------------


       0   1.00e-01        635       4.96
       1   3.16e+01        224       0.33
       2   5.62e+02          8       0.16
       3   1.33e+02         59       0.23
       4   2.74e+02         28       0.22
       5   3.92e+02         14       0.17
       6   3.28e+02         22       0.19
       7   3.59e+02         20       0.18

Found alpha=3.59e+02 yielding exactly 20 features
Total search time: 6.42s

Step 2: Interpreting selected neurons


Generating 20 interpretation(s) per neuron: 100%|██████████| 400/400 [01:05<00:00,  6.13it/s]



Step 3: Scoring Interpretations
Found 5600 cached items; annotating 34400 uncached items


Scoring neuron interpretation fidelity (20 neurons; 20 candidate interps per neuron; 100 examples to score each interp): 100%|██████████| 34400/34400 [21:41<00:00, 26.44it/s]



Most predictive features of Yelp reviews:


,neuron_idx,source_sae,target_lasso,interpretation,f1_fidelity_score
0,318,"(128, 64)",0.714656,"<think>\n\n</think>\n\n- ""mentions the presence of a soft-tissue mass in a specific anatomical location",0.959583
1,470,"(256, 64)",0.428413,"<think>\n\n</think>\n\n- ""mentions the presence of a tumor or cancer, including specific types such as carcinoma, sarcoma, or metastasis",0.969897
2,263,"(128, 64)",0.190897,"<think>\n\n</think>\n\n- ""mentions the presence of fever, elevated inflammatory markers (such as C-reactive protein or leukocytosis), and evidence of infection or abscess formation",0.980000
3,269,"(128, 64)",0.144958,"<think>\n\n</think>\n\n- ""describes postoperative or post-procedural complications requiring intensive care or advanced life support",0.901099
4,282,"(128, 64)",0.090335,"<think>\n\n</think>\n\n- ""describes the presence of spindle cells in histopathological findings",0.717949
5,283,"(128, 64)",0.075873,"<think>\n\n</think>\n\n- ""mentions adverse reactions or complications from medications or treatments",0.959583
6,568,"(256, 64)",0.062473,"<think>\n\n</think>\n\n- ""mentions a well-defined or localized lesion identified through diagnostic imaging (e.g., ultrasound, MRI, CT) that is non-malignant or of moderate suspicion, and is described with specific imaging characteristics such as hypoechoic",0.802439
7,183,"(256, 32)",0.058397,"<think>\n\n</think>\n\n- ""mentions tumor cells positive for epithelial membrane antigen (EMA)",0.333333
8,275,"(128, 64)",0.057049,"<think>\n\n</think>\n\n- ""mentions metastasis or recurrence of cancer in multiple anatomical locations",0.802892
9,296,"(128, 64)",0.021988,"<think>\n\n</think>\n\n- ""mentions neurological or psychiatric symptoms or conditions such as seizures, catatonia, hallucinations, or altered mental status",0.979592


**Save all in one json**

In [30]:
import os
import json
if not os.path.isdir(result_dir):
    os.makedirs(result_dir)
train_all_info_file = os.path.join(result_dir, "train_all_info.jsonl")
test_all_info_file = os.path.join(result_dir, "test_all_info.jsonl")
with open(test_all_info_file, 'w') as f:
    for text, label, idx in zip(test_texts, test_y, range(len(test_texts))):
        f.write(json.dumps({"idx": idx, "profile": text, "label": label}) + "\n")
with open(train_all_info_file, 'w') as f:
    for text, label, idx in zip(texts, labels, range(len(texts))):
        f.write(json.dumps({"idx": idx, "profile": text, "label": label}) + "\n")

In [31]:
from hypothesaes.sae import SparseAutoencoder, load_model, get_multiple_sae_activations, get_sae_checkpoint_name
activations, neuron_source_sae_info = get_multiple_sae_activations(sae_list, embeddings, return_neuron_source_info=True)
test_activations, neuron_source_sae_info_test = get_multiple_sae_activations(sae_list, test_embeddings, return_neuron_source_info=True)

Computing activations (batchsize=16384):   0%|          | 0/1 [00:00<?, ?it/s]

Computing activations (batchsize=16384): 100%|██████████| 1/1 [00:00<00:00, 894.50it/s]


In [32]:
def clean_interpretation(text):
    return text.replace("<think>", "").replace("</think>", "").strip().lstrip("-").strip().strip('"')

# 创建解释字典
if multi_class:
    neuron_interpretations = {
        row['neuron_idx']: clean_interpretation(row['interpretation'])
        for _, row in results.iterrows()
    }
else:
    neuron_interpretations = {}
    for key, value in results.items():
        neuron_interpretations[key] = {
            row['neuron_idx']: clean_interpretation(row['interpretation'])
            for _, row in value.iterrows()
        }

In [33]:
import torch
def describe_from_activations(activations: np.array, neuron_interpretations: dict, threshold=0.5, extra_info =None):
    """根据 activation 向量和解释字典，返回解释列表"""

    
    descriptions = []
    for idx, value in enumerate(activations):
        if value.item() > threshold and idx in neuron_interpretations:
            if extra_info is not None:
                descriptions.append((idx, neuron_interpretations[idx], value.item(), extra_info))
            else:
                descriptions.append((idx, neuron_interpretations[idx], value.item()))
    
    # 可以按激活值排序
    descriptions = sorted(descriptions, key=lambda x: -x[2])
    return descriptions
def render_descriptions(descriptions, multiclass=False):
    if not descriptions:
        return "No meaningful activations found."

    output = ["This text likely involves:"]
    if multiclass:   
        for idx, interp, score in descriptions:
            output.append(f"- {interp} (activation={score:.2f})")
    else:
        for idx, interp, score, type in descriptions:
            output.append(f"- {interp} (activation={score:.2f}, type={type})")
    return "\n".join(output)

In [34]:
interpret_list = []
if multi_class:
    for act in activations:
        descriptions = describe_from_activations(act, neuron_interpretations, threshold=0)
        interpret_list.append(render_descriptions(descriptions, True))
else:
    for act in activations:
        descriptions = []
        for key, value in neuron_interpretations.items():
            descriptions.extend(describe_from_activations(act, value, threshold=0, extra_info=f'[{key}]'))
        interpret_list.append(render_descriptions(descriptions, False))
with open(os.path.join(result_dir, 'test_interpretations.jsonl'), "w") as f:
    for text, interpret in zip(range(len(interpret_list)), interpret_list):
        f.write(json.dumps({"text": text, "interpretation": interpret}) + "\n")
interpret_list_test = []
if multi_class:
    for act in test_activations:
        descriptions = describe_from_activations(act, neuron_interpretations, threshold=0)
        interpret_list_test.append(render_descriptions(descriptions, True))
else:
    for act in test_activations:
        descriptions = []
        for key, value in neuron_interpretations.items():
            descriptions.extend(describe_from_activations(act, value, threshold=0, extra_info=f'[{key}]'))
        interpret_list_test.append(render_descriptions(descriptions, False))
with open(os.path.join(result_dir, 'test_interpretations.jsonl'), "w") as f:
    for text, interpret in zip(range(len(interpret_list_test)), interpret_list_test):
        f.write(json.dumps({"text": text, "interpretation": interpret}) + "\n")
    

**Evaluate held-out generalization**

Finally, we evaluate whether these are good hypotheses by testing whether their natural language interpretations can predict the target variable.  

We compute annotations for each hypothesized concept on a holdout set (not seen during SAE training & feature selection).

After annotation, we output a dataframe with the following columns:
- `hypothesis`: The natural language hypothesis (which came from interpreting a predictive neuron in the SAE)
- `separation_score`: How much the target variable differs when the concept is present vs. absent (i.e., $E[Y\mid\text{concept} = 1] - E[Y\mid\text{concept} = 0]$).
- `separation_pvalue`: The t-test p-value of the null hypothesis that the separation score is 0 (i.e., the concept is not associated with the target variable).
- `regression_coef`: The coefficient of the concept in a multivariate linear regression of the target variable on all concepts.
- `regression_pval`: The p-value of the null hypothesis that the regression coefficient is 0.
- `feature_prevalence`: The fraction of examples that contain the concept.

Additionally, we output the evaluation metrics used in the paper:
- Significant hypotheses: the number of hypotheses that are significant in the multivariate regression at a specified significance level (default $0.1$) after Bonferroni correction. You can pass in a different significance level using the `corrected_pval_threshold` parameter.
- AUC or $R^2$: how well the hypotheses collectively predict the target variable in the multivariate regression.


In [35]:
# holdout_df = pd.read_json(os.path.join(base_dir, "yelp-demo-holdout-2K.json"), lines=True)
# holdout_texts = holdout_df['text'].tolist()
# holdout_labels = holdout_df['stars'].values
from hypothesaes.quickstart import evaluate_hypotheses
if multi_class:
    metrics, evaluation_df = evaluate_hypotheses(
        hypotheses_df=results,
        texts=test_texts,
        labels=label_test,
        cache_name=CACHE_NAME,
        max_words_per_example=1024,
        annotator_model="Qwen/Qwen3-32B",  # Use the same model as for training
        classification=True
    )

    pd.set_option('display.max_colwidth', None)
    display(evaluation_df)
    pd.reset_option('display.max_colwidth')

    print("\nHoldout Set Metrics:")
    print(f"R² Score: {metrics['r2']:.3f}")
    print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
        f"(p < {metrics['Significant'][2]:.3e})")
else:
    metrics, evaluation_df = {}, {}
    for key in number_dict.keys():
        key_norm = key.replace("/", "or")
        key_norm = key_norm.replace(" ", "_")
        converted_key = {number_dict[dic_key]: 1 if dic_key == key else 0 for dic_key in number_dict.keys()}
        converted_label = [converted_key[label] for label in label_test]
        metrics[key], evaluation_df[key] = evaluate_hypotheses(
            hypotheses_df=results[f'{key}_vs_rest'],
            texts=test_texts,
            labels=converted_label,
            cache_name=os.path.join(CACHE_NAME, CACHE_SIGNAL, f"{key}_vs_rest"),
            max_words_per_example=1024,
            annotator_model="Qwen/Qwen3-32B",  # Use the same model as for training
            classification=True
        )
        evaluation_df[key].to_csv(os.path.join(result_dir, f"evaluation_{key_norm}.csv"), index=False)
        with open(os.path.join(result_dir, f"metrics_{key_norm}.json"), "w") as f:
            json.dump(metrics[key], f, indent=4)
        pd.set_option('display.max_colwidth', None)
        display(evaluation_df[key])
        pd.reset_option('display.max_colwidth')
        print(f"\nHoldout Set Metrics for {key}:")
        print(f"R² Score: {metrics[key]['r2']:.3f}")
        print(f"Significant hypotheses: {metrics[key]['Significant'][0]}/{metrics[key]['Significant'][1]} " 
            f"(p < {metrics[key]['Significant'][2]:.3e})")


Step 1: Annotating texts with 20 hypotheses
Found 1872 cached items; annotating 7488 uncached items


Annotating: 100%|██████████| 7488/7488 [05:26<00:00, 22.90it/s]


Step 2: Computing predictiveness of hypothesis annotations
         Current function value: 0.460162
         Iterations: 35


/home/sevan/anaconda3/envs/llm/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/home/sevan/anaconda3/envs/llm/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
0,"<think>\n\n</think>\n\n- ""mentions the presence of a soft-tissue mass in a specific anatomical location",0.508093,1.813360e-29,1.599666,5.177103e-08,0.363248
13,"<think>\n\n</think>\n\n- ""mentions a well-defined radiolucent lesion in the mandible",0.506494,1.363294e-02,23.009824,9.994364e-01,0.012821
12,"<think>\n\n</think>\n\n- ""mentions severe hypercholesterolemia or very high LDL cholesterol levels",0.501071,3.178295e-01,18.729073,9.993001e-01,0.002137
10,"<think>\n\n</think>\n\n- ""mentions a well-defined or intramedullary lesion observed via MRI or CT imaging",0.427972,7.401551e-19,0.698302,4.233283e-02,0.305556
1,"<think>\n\n</think>\n\n- ""mentions the presence of a tumor or cancer, including specific types such as carcinoma, sarcoma, or metastasis",0.421296,2.398248e-15,1.355118,2.402121e-04,0.230769
17,"<think>\n\n</think>\n\n- ""mentions a lesion or swelling in the mandible, often associated with radiographic findings such as radiolucency, cortical expansion, or root resorption",0.408734,1.048434e-02,0.374787,7.985896e-01,0.021368
4,"<think>\n\n</think>\n\n- ""describes the presence of spindle cells in histopathological findings",0.404444,7.334529e-04,0.788322,3.884740e-01,0.038462
8,"<think>\n\n</think>\n\n- ""mentions metastasis or recurrence of cancer in multiple anatomical locations",0.362737,8.672319e-05,0.926857,1.478815e-01,0.066239
6,"<think>\n\n</think>\n\n- ""mentions a well-defined or localized lesion identified through diagnostic imaging (e.g., ultrasound, MRI, CT) that is non-malignant or of moderate suspicion, and is described with specific imaging characteristics such as hypoechoic",0.361305,4.734442e-09,0.315924,4.513400e-01,0.162393
18,"<think>\n\n</think>\n\n- ""describes a well-defined or localized lesion identified through MRI with specific signal characteristics on T1- and T2-weighted images",0.356143,1.279230e-09,0.550124,1.697451e-01,0.183761



Holdout Set Metrics:
R² Score: 0.336
Significant hypotheses: 2/20 (p < 5.000e-03)


In [36]:
if multi_class:
        pd.set_option('display.max_colwidth', None)
        display(evaluation_df)
        pd.reset_option('display.max_colwidth')
        print("\nHoldout Set Metrics:")
        print(f"R² Score: {metrics['r2']:.3f}")
        print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
            f"(p < {metrics['Significant'][2]:.3e})")
        evaluation_df.to_csv(os.path.join(result_dir, "evaluation.csv"), index=False)
        with open(os.path.join(result_dir, "metrics.json"), "w") as f:
            json.dump(metrics, f, indent=4)
        
else:
        for key in metrics.keys():
                # print(metrics[key])
                pd.set_option('display.max_colwidth', None)
                display(evaluation_df[key])
                pd.reset_option('display.max_colwidth')
                print(f"\nHoldout Set Metrics for {key}:")
                print(f"R² Score: {metrics[key]['r2']:.3f}")
                print(f"Significant hypotheses: {metrics[key]['Significant'][0]}/{metrics[key]['Significant'][1]} " 
                f"(p < {metrics[key]['Significant'][2]:.3e})")
                evaluation_df[key].to_csv(os.path.join(result_dir, f"evaluation_{key}.csv"), index=False)
                with open(os.path.join(result_dir, f"metrics_{key}.json"), "w") as f:
                    json.dump(metrics[key], f, indent=4)

,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
0,"<think>\n\n</think>\n\n- ""mentions the presence of a soft-tissue mass in a specific anatomical location",0.508093,1.813360e-29,1.599666,5.177103e-08,0.363248
13,"<think>\n\n</think>\n\n- ""mentions a well-defined radiolucent lesion in the mandible",0.506494,1.363294e-02,23.009824,9.994364e-01,0.012821
12,"<think>\n\n</think>\n\n- ""mentions severe hypercholesterolemia or very high LDL cholesterol levels",0.501071,3.178295e-01,18.729073,9.993001e-01,0.002137
10,"<think>\n\n</think>\n\n- ""mentions a well-defined or intramedullary lesion observed via MRI or CT imaging",0.427972,7.401551e-19,0.698302,4.233283e-02,0.305556
1,"<think>\n\n</think>\n\n- ""mentions the presence of a tumor or cancer, including specific types such as carcinoma, sarcoma, or metastasis",0.421296,2.398248e-15,1.355118,2.402121e-04,0.230769
17,"<think>\n\n</think>\n\n- ""mentions a lesion or swelling in the mandible, often associated with radiographic findings such as radiolucency, cortical expansion, or root resorption",0.408734,1.048434e-02,0.374787,7.985896e-01,0.021368
4,"<think>\n\n</think>\n\n- ""describes the presence of spindle cells in histopathological findings",0.404444,7.334529e-04,0.788322,3.884740e-01,0.038462
8,"<think>\n\n</think>\n\n- ""mentions metastasis or recurrence of cancer in multiple anatomical locations",0.362737,8.672319e-05,0.926857,1.478815e-01,0.066239
6,"<think>\n\n</think>\n\n- ""mentions a well-defined or localized lesion identified through diagnostic imaging (e.g., ultrasound, MRI, CT) that is non-malignant or of moderate suspicion, and is described with specific imaging characteristics such as hypoechoic",0.361305,4.734442e-09,0.315924,4.513400e-01,0.162393
18,"<think>\n\n</think>\n\n- ""describes a well-defined or localized lesion identified through MRI with specific signal characteristics on T1- and T2-weighted images",0.356143,1.279230e-09,0.550124,1.697451e-01,0.183761



Holdout Set Metrics:
R² Score: 0.336
Significant hypotheses: 2/20 (p < 5.000e-03)


In [37]:
pd.set_option('display.max_colwidth', None)
display(evaluation_df)
pd.reset_option('display.max_colwidth')
print("\nHoldout Set Metrics:")
# print(f"R² Score: {metrics['r2']:.3f}")
print(f"Significant hypotheses: {metrics['Significant'][0]}/{metrics['Significant'][1]} " 
    f"(p < {metrics['Significant'][2]:.3e})")
evaluation_df.to_csv(os.path.join(result_dir, "evaluation.csv"), index=False)
with open(os.path.join(result_dir, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

,hypothesis,separation_score,separation_pval,regression_coef,regression_pval,feature_prevalence
0,"<think>\n\n</think>\n\n- ""mentions the presence of a soft-tissue mass in a specific anatomical location",0.508093,1.813360e-29,1.599666,5.177103e-08,0.363248
13,"<think>\n\n</think>\n\n- ""mentions a well-defined radiolucent lesion in the mandible",0.506494,1.363294e-02,23.009824,9.994364e-01,0.012821
12,"<think>\n\n</think>\n\n- ""mentions severe hypercholesterolemia or very high LDL cholesterol levels",0.501071,3.178295e-01,18.729073,9.993001e-01,0.002137
10,"<think>\n\n</think>\n\n- ""mentions a well-defined or intramedullary lesion observed via MRI or CT imaging",0.427972,7.401551e-19,0.698302,4.233283e-02,0.305556
1,"<think>\n\n</think>\n\n- ""mentions the presence of a tumor or cancer, including specific types such as carcinoma, sarcoma, or metastasis",0.421296,2.398248e-15,1.355118,2.402121e-04,0.230769
17,"<think>\n\n</think>\n\n- ""mentions a lesion or swelling in the mandible, often associated with radiographic findings such as radiolucency, cortical expansion, or root resorption",0.408734,1.048434e-02,0.374787,7.985896e-01,0.021368
4,"<think>\n\n</think>\n\n- ""describes the presence of spindle cells in histopathological findings",0.404444,7.334529e-04,0.788322,3.884740e-01,0.038462
8,"<think>\n\n</think>\n\n- ""mentions metastasis or recurrence of cancer in multiple anatomical locations",0.362737,8.672319e-05,0.926857,1.478815e-01,0.066239
6,"<think>\n\n</think>\n\n- ""mentions a well-defined or localized lesion identified through diagnostic imaging (e.g., ultrasound, MRI, CT) that is non-malignant or of moderate suspicion, and is described with specific imaging characteristics such as hypoechoic",0.361305,4.734442e-09,0.315924,4.513400e-01,0.162393
18,"<think>\n\n</think>\n\n- ""describes a well-defined or localized lesion identified through MRI with specific signal characteristics on T1- and T2-weighted images",0.356143,1.279230e-09,0.550124,1.697451e-01,0.183761



Holdout Set Metrics:
Significant hypotheses: 2/20 (p < 5.000e-03)


In [38]:
evaluation_df
with open(os.path.join(result_dir, 'hypotheses_info.jsonl'), "w") as f:
    for _, row in evaluation_df.iterrows():
        f.write(json.dumps(row.to_dict()) + "\n")

In [39]:
#output hypothesis and its coef in csv
df_coef = evaluation_df[['hypothesis', 'regression_coef']]
df_coef.to_csv(os.path.join(result_dir, 'hypothesis_coef.csv'), index=False)